# Convolution on the GPU (chapter 7)

Study guide for the convolution chapter: constant memory, caches, and shared-memory tiles with halo cells. You already know what a convolution computes, so this guide is about how to make a GPU compute it fast and why each version is faster than the last. It goes with session 15 (`15_convolution`).

This notebook runs the **complete programs**: every kernel of the chapter on large inputs, compared side by side with timings, and the **bike-sharing examples**, which apply the same pattern to another domain to give you ideas for your project. Do the step-by-step notebook (`colab_1_step_by_step.ipynb`) first. Timings are only meaningful on the GPU.

The full theory, the exercises and their solutions are in `ec2/05_conv/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## `bikeshare_demand_heatmap.cu`

rush-hour demand heatmap of a bike-sharing city with the tiled constant-memory kernel: hot zones for new stations and rebalancing

This is a bike-sharing example: the README section *Ideas for your project* explains it and lists where the same pattern may appear in your project.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile bikeshare_demand_heatmap.cu
#include "cuda_compat.h"

#include <cmath>
#include <cstdint>
#include <cstdio>
#include <vector>

#ifndef CPU_RIDE_REQUESTS
#define CPU_RIDE_REQUESTS 40000
#endif

const int RIDE_REQUEST_COUNT = BY_BACKEND(2000000, CPU_RIDE_REQUESTS);

const int MAP_SIDE_METRES = 6000;
const int GRID_SIDE = 125;
const int CELL_METRES = MAP_SIDE_METRES / GRID_SIDE;

#define FILTER_RADIUS 6
#define FILTER_DIM (2 * FILTER_RADIUS + 1)
#define IN_TILE_DIM 32
#define OUT_TILE_DIM (IN_TILE_DIM - 2 * FILTER_RADIUS)

const double GAUSSIAN_SIGMA_CELLS = 3.0;

__constant__ float G_c[FILTER_DIM][FILTER_DIM];

__global__ void smooth_tiled_kernel(const float* H, float* S, int width, int height) {
  int col = (int)(blockIdx.x * OUT_TILE_DIM) + (int)threadIdx.x - FILTER_RADIUS;
  int row = (int)(blockIdx.y * OUT_TILE_DIM) + (int)threadIdx.y - FILTER_RADIUS;

  __shared__ float H_s[IN_TILE_DIM][IN_TILE_DIM];
  if (row >= 0 && row < height && col >= 0 && col < width) {
    H_s[threadIdx.y][threadIdx.x] = H[row * width + col];
  } else {
    H_s[threadIdx.y][threadIdx.x] = 0.0f;
  }
  __syncthreads();

  int tileCol = (int)threadIdx.x - FILTER_RADIUS;
  int tileRow = (int)threadIdx.y - FILTER_RADIUS;
  bool insideImage = col >= 0 && col < width && row >= 0 && row < height;
  bool computesOutput =
      tileCol >= 0 && tileCol < OUT_TILE_DIM && tileRow >= 0 && tileRow < OUT_TILE_DIM;

  if (insideImage && computesOutput) {
    float s = 0.0f;
    for (int fRow = 0; fRow < FILTER_DIM; fRow++) {
      for (int fCol = 0; fCol < FILTER_DIM; fCol++) {
        s += G_c[fRow][fCol] * H_s[tileRow + fRow][tileCol + fCol];
      }
    }
    S[row * width + col] = s;
  }
}

using Grid = std::vector<float>;

static void smooth_cpu(const Grid& H, const std::vector<float>& G, Grid& S, int width, int height) {
  for (int row = 0; row < height; ++row) {
    for (int col = 0; col < width; ++col) {
      float s = 0.0f;
      for (int fRow = 0; fRow < FILTER_DIM; ++fRow) {
        for (int fCol = 0; fCol < FILTER_DIM; ++fCol) {
          int inRow = row - FILTER_RADIUS + fRow;
          int inCol = col - FILTER_RADIUS + fCol;
          if (inRow >= 0 && inRow < height && inCol >= 0 && inCol < width) {
            s += G[fRow * FILTER_DIM + fCol] * H[inRow * width + inCol];
          }
        }
      }
      S[row * width + col] = s;
    }
  }
}

class SplitMix64 {
 public:
  explicit SplitMix64(uint64_t seed) : state_(seed) {
  }

  uint64_t next() {
    state_ += GOLDEN_RATIO_INCREMENT;
    uint64_t z = state_;
    z = (z ^ (z >> 30)) * FIRST_MIX_MULTIPLIER;
    z = (z ^ (z >> 27)) * SECOND_MIX_MULTIPLIER;
    return z ^ (z >> 31);
  }

  int below(int n) {
    return static_cast<int>(next() % static_cast<uint64_t>(n));
  }

  double uniform() {
    return static_cast<double>(next() >> UNIFORM_SHIFT) * UNIFORM_SCALE;
  }

  double standardNormal() {
    const double awayFromZero = 1.0 - uniform();
    const double angle = TWO_PI * uniform();
    return std::sqrt(-2.0 * std::log(awayFromZero)) * std::cos(angle);
  }

 private:
  static constexpr uint64_t GOLDEN_RATIO_INCREMENT = 0x9E3779B97F4A7C15ull;
  static constexpr uint64_t FIRST_MIX_MULTIPLIER = 0xBF58476D1CE4E5B9ull;
  static constexpr uint64_t SECOND_MIX_MULTIPLIER = 0x94D049BB133111EBull;
  static constexpr int UNIFORM_SHIFT = 11;
  static constexpr double UNIFORM_SCALE = 1.0 / 9007199254740992.0;
  static constexpr double TWO_PI = 6.283185307179586;

  uint64_t state_;
};

struct Position {
  int x;
  int y;
};

struct Station {
  int id;
  Position position;
  int docks;
  int bikesDocked;
};

const int STATION_COUNT = 48;
const uint64_t STATION_SEED = 48;
const int STATION_MARGIN_METRES = 300;
const int MIN_DOCKS = 12;
const int EXTRA_DOCKS_RANGE = 13;

enum class Rush { Morning, Evening };

struct DemandCluster {
  const char* kind;
  Position centre;
  double spreadMetres;
  double morningWeight;
  double eveningWeight;
};

const DemandCluster CLUSTERS[] = {
    {"homes (north-west)", {1200, 4800}, 260, 5.0, 0.8},
    {"homes (south)", {2500, 900}, 300, 4.0, 0.6},
    {"condos (east)", {5000, 3900}, 200, 3.0, 1.0},
    {"offices (centre)", {3100, 3000}, 200, 0.8, 5.0},
    {"university (west)", {900, 2500}, 220, 1.5, 3.5},
    {"market (south-east)", {4700, 1500}, 250, 1.0, 2.5},
};
const int CLUSTER_COUNT = sizeof(CLUSTERS) / sizeof(CLUSTERS[0]);

const uint64_t RIDE_REQUEST_SEED = 2024;
const double MORNING_REQUEST_SHARE = 0.45;
const double BACKGROUND_REQUEST_SHARE = 0.15;
const double MORNING_RUSH_HOUR = 8.0;
const double EVENING_RUSH_HOUR = 18.0;
const double RUSH_SPREAD_HOURS = 0.5;
const int SECONDS_PER_MINUTE = 60;
const int MINUTES_PER_HOUR = 60;
const int SECONDS_PER_HOUR = SECONDS_PER_MINUTE * MINUTES_PER_HOUR;
const int MS_PER_SECOND = 1000;
const double MS_PER_HOUR = static_cast<double>(SECONDS_PER_HOUR) * MS_PER_SECOND;

const int REQUESTS_TO_SHOW = 3;
const int HOT_ZONE_COUNT = 5;
const int HOT_ZONE_MIN_SEPARATION_CELLS = 15;
const double NEARBY_REQUEST_RADIUS_METRES = 250.0;
const double NEW_STATION_DISTANCE_METRES = 500.0;
const float SUPPRESSED_CELL = -1.0f;
const double RELATIVE_ERROR_FLOOR = 1e-3;
const double MAX_RELATIVE_ERROR = 1e-4;

struct RideRequest {
  int id;
  int timeMs;
  Position origin;
};

struct TimeWindow {
  const char* name;
  double fromHour;
  double toHour;
};

const TimeWindow WINDOWS[] = {
    {"morning rush 07:00-09:00", 7.0, 9.0},
    {"evening rush 17:00-19:00", 17.0, 19.0},
};

struct HotZone {
  int row;
  int col;
  float value;
};

static int cellOf(int metres) {
  return imin(GRID_SIDE - 1, metres / CELL_METRES);
}

static int cellCentreMetres(int cell) {
  return cell * CELL_METRES + CELL_METRES / 2;
}

static double distanceMetres(Position a, Position b) {
  return std::hypot(static_cast<double>(a.x - b.x), static_cast<double>(a.y - b.y));
}

static bool insideMap(Position position) {
  return position.x >= 0 && position.x < MAP_SIDE_METRES && position.y >= 0 &&
         position.y < MAP_SIDE_METRES;
}

static double requestHour(const RideRequest& request) {
  return request.timeMs / MS_PER_HOUR;
}

static bool inWindow(const RideRequest& request, const TimeWindow& window) {
  const double hour = requestHour(request);
  return hour >= window.fromHour && hour < window.toHour;
}

static double demandWeight(const DemandCluster& cluster, Rush rush) {
  return rush == Rush::Morning ? cluster.morningWeight : cluster.eveningWeight;
}

static Position randomPosition(SplitMix64& rng, int marginMetres) {
  const int span = MAP_SIDE_METRES - 2 * marginMetres;
  Position position;
  position.x = marginMetres + rng.below(span);
  position.y = marginMetres + rng.below(span);
  return position;
}

static std::vector<Station> makeStations() {
  SplitMix64 rng(STATION_SEED);
  std::vector<Station> stations(STATION_COUNT);

  for (int id = 0; id < STATION_COUNT; ++id) {
    stations[id].id = id;
    stations[id].position = randomPosition(rng, STATION_MARGIN_METRES);
  }

  for (Station& station : stations) {
    station.docks = MIN_DOCKS + rng.below(EXTRA_DOCKS_RANGE);
    station.bikesDocked = rng.below(station.docks + 1);
  }

  return stations;
}

class RideRequestGenerator {
 public:
  explicit RideRequestGenerator(uint64_t seed) : rng_(seed) {
  }

  std::vector<RideRequest> makeRequests(int count) {
    std::vector<RideRequest> requests;
    requests.reserve(count);

    while (static_cast<int>(requests.size()) < count) {
      const Rush rush = rng_.uniform() < MORNING_REQUEST_SHARE ? Rush::Morning : Rush::Evening;
      const double hour = requestTimeHour(rush);
      const Position origin = requestOrigin(rush);
      if (!insideMap(origin)) {
        continue;
      }

      RideRequest request;
      request.id = static_cast<int>(requests.size());
      request.timeMs = static_cast<int>(hour * MS_PER_HOUR);
      request.origin = origin;
      requests.push_back(request);
    }

    return requests;
  }

 private:
  double requestTimeHour(Rush rush) {
    const double peak = rush == Rush::Morning ? MORNING_RUSH_HOUR : EVENING_RUSH_HOUR;
    return peak + RUSH_SPREAD_HOURS * rng_.standardNormal();
  }

  Position requestOrigin(Rush rush) {
    if (rng_.uniform() < BACKGROUND_REQUEST_SHARE) {
      return randomPosition(rng_, 0);
    }
    return nearCluster(CLUSTERS[pickCluster(rush)]);
  }

  int pickCluster(Rush rush) {
    double totalWeight = 0.0;
    for (const DemandCluster& cluster : CLUSTERS) {
      totalWeight += demandWeight(cluster, rush);
    }

    double pick = totalWeight * rng_.uniform();
    int k = 0;
    while (k < CLUSTER_COUNT - 1 && pick > demandWeight(CLUSTERS[k], rush)) {
      pick -= demandWeight(CLUSTERS[k], rush);
      ++k;
    }
    return k;
  }

  Position nearCluster(const DemandCluster& cluster) {
    Position position;
    position.x = cluster.centre.x + static_cast<int>(rng_.standardNormal() * cluster.spreadMetres);
    position.y = cluster.centre.y + static_cast<int>(rng_.standardNormal() * cluster.spreadMetres);
    return position;
  }

  SplitMix64 rng_;
};

static int indexOfMaximum(const Grid& grid) {
  int best = 0;
  for (int i = 1; i < static_cast<int>(grid.size()); ++i) {
    if (grid[i] > grid[best]) {
      best = i;
    }
  }
  return best;
}

static void suppressAround(Grid& grid, int centreRow, int centreCol, int radius) {
  const int firstRow = imax(0, centreRow - radius);
  const int lastRow = imin(GRID_SIDE - 1, centreRow + radius);
  const int firstCol = imax(0, centreCol - radius);
  const int lastCol = imin(GRID_SIDE - 1, centreCol + radius);

  for (int row = firstRow; row <= lastRow; ++row) {
    for (int col = firstCol; col <= lastCol; ++col) {
      grid[row * GRID_SIDE + col] = SUPPRESSED_CELL;
    }
  }
}

static std::vector<HotZone> hotZones(const Grid& smoothed, int count, int minSeparation) {
  Grid candidates = smoothed;
  std::vector<HotZone> zones;

  for (int n = 0; n < count; ++n) {
    const int best = indexOfMaximum(candidates);
    const int row = best / GRID_SIDE;
    const int col = best % GRID_SIDE;
    zones.push_back({row, col, smoothed[best]});
    suppressAround(candidates, row, col, minSeparation);
  }

  return zones;
}

static std::vector<float> gaussianWeights() {
  std::vector<float> weights(FILTER_DIM * FILTER_DIM);
  const double twoSigmaSquared = 2 * GAUSSIAN_SIGMA_CELLS * GAUSSIAN_SIGMA_CELLS;
  double sum = 0.0;

  for (int i = 0; i < FILTER_DIM; ++i) {
    for (int j = 0; j < FILTER_DIM; ++j) {
      double dy = i - FILTER_RADIUS;
      double dx = j - FILTER_RADIUS;
      weights[i * FILTER_DIM + j] = (float)std::exp(-(dx * dx + dy * dy) / twoSigmaSquared);
      sum += weights[i * FILTER_DIM + j];
    }
  }

  for (float& weight : weights) {
    weight = (float)(weight / sum);
  }
  return weights;
}

static Grid binRequests(const std::vector<RideRequest>& requests,
                        const TimeWindow& window,
                        int& binnedRequests) {
  Grid histogram(static_cast<size_t>(GRID_SIDE) * GRID_SIDE, 0.0f);
  binnedRequests = 0;

  for (const RideRequest& request : requests) {
    if (!inWindow(request, window)) {
      continue;
    }
    const int cell = cellOf(request.origin.y) * GRID_SIDE + cellOf(request.origin.x);
    histogram[cell] += 1.0f;
    ++binnedRequests;
  }

  return histogram;
}

static int requestsNear(const std::vector<RideRequest>& requests,
                        const TimeWindow& window,
                        Position centre) {
  int count = 0;
  for (const RideRequest& request : requests) {
    if (inWindow(request, window) &&
        distanceMetres(centre, request.origin) <= NEARBY_REQUEST_RADIUS_METRES) {
      ++count;
    }
  }
  return count;
}

static const Station& nearestStation(const std::vector<Station>& stations, Position position) {
  const Station* nearest = &stations[0];
  for (const Station& station : stations) {
    if (distanceMetres(position, station.position) < distanceMetres(position, nearest->position)) {
      nearest = &station;
    }
  }
  return *nearest;
}

static double maxRelativeError(const Grid& expected, const Grid& actual) {
  double worst = 0.0;
  for (size_t i = 0; i < expected.size(); ++i) {
    double difference = std::fabs((double)expected[i] - actual[i]);
    worst = std::fmax(worst, difference / (RELATIVE_ERROR_FLOOR + std::fabs((double)expected[i])));
  }
  return worst;
}

class DeviceSmoother {
 public:
  DeviceSmoother()
      : bytes_(static_cast<size_t>(GRID_SIDE) * GRID_SIDE * sizeof(float)),
        block_(IN_TILE_DIM, IN_TILE_DIM),
        grid_(cdiv(GRID_SIDE, OUT_TILE_DIM), cdiv(GRID_SIDE, OUT_TILE_DIM)) {
    CHECK(cudaMalloc(&histogram_, bytes_));
    CHECK(cudaMalloc(&smoothed_, bytes_));
  }

  ~DeviceSmoother() {
    CHECK(cudaFree(histogram_));
    CHECK(cudaFree(smoothed_));
  }

  void uploadWeights(const std::vector<float>& weights) {
    CHECK(cudaMemcpyToSymbol(G_c, weights.data(), weights.size() * sizeof(float)));
  }

  Grid smooth(const Grid& histogram, double& kernelMs) {
    CHECK(cudaMemcpy(histogram_, histogram.data(), bytes_, cudaMemcpyHostToDevice));

    Timer timer;
    launch(smooth_tiled_kernel,
           grid_,
           block_,
           (const float*)histogram_,
           smoothed_,
           GRID_SIDE,
           GRID_SIDE);
    CHECK(cudaGetLastError());
    kernelMs = timer.ms();

    Grid smoothed(static_cast<size_t>(GRID_SIDE) * GRID_SIDE);
    CHECK(cudaMemcpy(smoothed.data(), smoothed_, bytes_, cudaMemcpyDeviceToHost));
    return smoothed;
  }

 private:
  size_t bytes_;
  dim3 block_;
  dim3 grid_;
  float* histogram_;
  float* smoothed_;
};

static void printHeader() {
  std::printf("Bike-share demand heatmap, %d x %d cells of %d m, %dx%d Gaussian, backend: %s\n",
              GRID_SIDE,
              GRID_SIDE,
              CELL_METRES,
              FILTER_DIM,
              FILTER_DIM,
              CUDA_BACKEND);
  std::printf("map: %d m x %d m, x grows to the east and y to the north, %d stations\n",
              MAP_SIDE_METRES,
              MAP_SIDE_METRES,
              STATION_COUNT);
}

static void printClock(int timeMs) {
  const int totalSeconds = timeMs / MS_PER_SECOND;
  const int hours = totalSeconds / SECONDS_PER_HOUR;
  const int minutes = totalSeconds / SECONDS_PER_MINUTE % MINUTES_PER_HOUR;
  const int seconds = totalSeconds % SECONDS_PER_MINUTE;
  std::printf("%02d:%02d:%02d", hours, minutes, seconds);
}

static void printFirstRequests(const std::vector<RideRequest>& requests) {
  std::printf("\n%zu synthetic ride requests (where a rider opened the app). First ones:\n",
              requests.size());
  for (int i = 0; i < REQUESTS_TO_SHOW; ++i) {
    std::printf("  request %d at ", requests[i].id);
    printClock(requests[i].timeMs);
    std::printf(" from (%d m, %d m)\n", requests[i].origin.x, requests[i].origin.y);
  }
}

static bool isLessThanHalfFull(const Station& station) {
  return 2 * station.bikesDocked < station.docks;
}

static void printSuggestion(const Station& station, double distance) {
  if (distance > NEW_STATION_DISTANCE_METRES) {
    std::printf("new station here (s%d is %.0f m away)\n", station.id, distance);
    return;
  }
  std::printf("s%d at %.0f m holds %d of %d bikes: %s\n",
              station.id,
              distance,
              station.bikesDocked,
              station.docks,
              isLessThanHalfFull(station) ? "bring bikes" : "keep it stocked");
}

static void printHotZone(int rank,
                         const HotZone& zone,
                         Position centre,
                         int nearbyRequests,
                         const std::vector<Station>& stations) {
  const Station& nearest = nearestStation(stations, centre);
  std::printf("   %4d  (%3d,%3d)    %5d  %5d   %9.3f   %8d       ",
              rank,
              zone.row,
              zone.col,
              centre.x,
              centre.y,
              zone.value,
              nearbyRequests);
  printSuggestion(nearest, distanceMetres(centre, nearest.position));
}

static void printHotZones(const std::vector<RideRequest>& requests,
                          const TimeWindow& window,
                          const Grid& smoothed,
                          const std::vector<Station>& stations) {
  std::printf("   rank  cell(row,col)  x (m)  y (m)  requests/cell  within 250 m  suggestion\n");

  const std::vector<HotZone> zones =
      hotZones(smoothed, HOT_ZONE_COUNT, HOT_ZONE_MIN_SEPARATION_CELLS);
  for (size_t k = 0; k < zones.size(); ++k) {
    const Position centre = {cellCentreMetres(zones[k].col), cellCentreMetres(zones[k].row)};
    printHotZone(static_cast<int>(k) + 1,
                 zones[k],
                 centre,
                 requestsNear(requests, window, centre),
                 stations);
  }
}

struct WindowInputs {
  const std::vector<RideRequest>& requests;
  const std::vector<Station>& stations;
  const std::vector<float>& weights;
};

static bool analyseWindow(const TimeWindow& window,
                          const WindowInputs& inputs,
                          DeviceSmoother& smoother) {
  int windowRequests = 0;
  const Grid histogram = binRequests(inputs.requests, window, windowRequests);

  double kernelMs = 0.0;
  const Grid smoothed = smoother.smooth(histogram, kernelMs);

  Grid reference(static_cast<size_t>(GRID_SIDE) * GRID_SIDE);
  Timer cpuTimer;
  smooth_cpu(histogram, inputs.weights, reference, GRID_SIDE, GRID_SIDE);
  const double cpuMs = cpuTimer.ms();

  const double error = maxRelativeError(reference, smoothed);
  const bool passed = error < MAX_RELATIVE_ERROR;

  const int busiestRawCell = indexOfMaximum(histogram);
  std::printf("\n== %s: %d requests  (kernel %.3f ms, CPU %.3f ms, %s, max rel err %.1e)\n",
              window.name,
              windowRequests,
              kernelMs,
              cpuMs,
              passed ? "PASS" : "FAIL",
              error);
  std::printf(
      "   busiest raw cell: (%d,%d) with %.0f requests (single cells are noisy; the ranking "
      "below uses the smoothed map)\n",
      busiestRawCell / GRID_SIDE,
      busiestRawCell % GRID_SIDE,
      histogram[busiestRawCell]);
  printHotZones(inputs.requests, window, smoothed, inputs.stations);

  return passed;
}

static bool analyseEveryWindow(const WindowInputs& inputs) {
  DeviceSmoother smoother;
  smoother.uploadWeights(inputs.weights);

  bool allPassed = true;
  for (const TimeWindow& window : WINDOWS) {
    allPassed = analyseWindow(window, inputs, smoother) && allPassed;
  }
  return allPassed;
}

int main() {
  printHeader();

  const std::vector<Station> stations = makeStations();
  RideRequestGenerator generator(RIDE_REQUEST_SEED);
  const std::vector<RideRequest> requests = generator.makeRequests(RIDE_REQUEST_COUNT);
  printFirstRequests(requests);

  const std::vector<float> weights = gaussianWeights();
  const WindowInputs inputs = {requests, stations, weights};
  const bool allPassed = analyseEveryWindow(inputs);

  std::printf("\nBefore each rush, trucks should fill the stations next to its rank-1 zones.\n");
  std::printf("%s\n", allPassed ? "ALL PASS" : "SOME TESTS FAILED");
  return allPassed ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. bikeshare_demand_heatmap.cu -o bikeshare_demand_heatmap && ./bikeshare_demand_heatmap

## `conv.cu`

kernels (a), (a'), (b), (c), (d), CPU reference, PASS/FAIL, timing table, intensity table

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile conv.cu
#include "cuda_compat.h"

#include <cmath>
#include <cstdio>
#include <random>
#include <vector>

#define FILTER_RADIUS 2
#define FILTER_DIM (2 * FILTER_RADIUS + 1)

#define IN_TILE_DIM 32
#define OUT_TILE_DIM (IN_TILE_DIM - 2 * FILTER_RADIUS)

#define TILE_DIM 32

__constant__ float F_c[FILTER_DIM][FILTER_DIM];

struct Filter {
  float w[FILTER_DIM][FILTER_DIM];
};

__global__ void convolution_2D_basic_kernel(
    const float* N, const float* F, float* P, int r, int width, int height) {
  int outCol = (int)(blockIdx.x * blockDim.x + threadIdx.x);
  int outRow = (int)(blockIdx.y * blockDim.y + threadIdx.y);

  if (outRow < height && outCol < width) {
    float Pvalue = 0.0f;
    for (int fRow = 0; fRow < 2 * r + 1; fRow++) {
      for (int fCol = 0; fCol < 2 * r + 1; fCol++) {
        int inRow = outRow - r + fRow;
        int inCol = outCol - r + fCol;
        if (inRow >= 0 && inRow < height && inCol >= 0 && inCol < width) {
          Pvalue += F[fRow * (2 * r + 1) + fCol] * N[inRow * width + inCol];
        }
      }
    }
    P[outRow * width + outCol] = Pvalue;
  }
}

__global__ void convolution_2D_struct_kernel(
    const float* N, Filter f, float* P, int width, int height) {
  int outCol = (int)(blockIdx.x * blockDim.x + threadIdx.x);
  int outRow = (int)(blockIdx.y * blockDim.y + threadIdx.y);

  if (outRow < height && outCol < width) {
    float Pvalue = 0.0f;
    for (int fRow = 0; fRow < FILTER_DIM; fRow++) {
      for (int fCol = 0; fCol < FILTER_DIM; fCol++) {
        int inRow = outRow - FILTER_RADIUS + fRow;
        int inCol = outCol - FILTER_RADIUS + fCol;
        if (inRow >= 0 && inRow < height && inCol >= 0 && inCol < width) {
          Pvalue += f.w[fRow][fCol] * N[inRow * width + inCol];
        }
      }
    }
    P[outRow * width + outCol] = Pvalue;
  }
}

__global__ void convolution_2D_const_mem_kernel(
    const float* N, float* P, int r, int width, int height) {
  int outCol = (int)(blockIdx.x * blockDim.x + threadIdx.x);
  int outRow = (int)(blockIdx.y * blockDim.y + threadIdx.y);

  if (outRow < height && outCol < width) {
    float Pvalue = 0.0f;
    for (int fRow = 0; fRow < 2 * r + 1; fRow++) {
      for (int fCol = 0; fCol < 2 * r + 1; fCol++) {
        int inRow = outRow - r + fRow;
        int inCol = outCol - r + fCol;
        if (inRow >= 0 && inRow < height && inCol >= 0 && inCol < width) {
          Pvalue += F_c[fRow][fCol] * N[inRow * width + inCol];
        }
      }
    }
    P[outRow * width + outCol] = Pvalue;
  }
}

__global__ void convolution_tiled_2D_const_mem_kernel(const float* N,
                                                      float* P,
                                                      int width,
                                                      int height) {
  int col = (int)(blockIdx.x * OUT_TILE_DIM) + (int)threadIdx.x - FILTER_RADIUS;
  int row = (int)(blockIdx.y * OUT_TILE_DIM) + (int)threadIdx.y - FILTER_RADIUS;

  __shared__ float N_s[IN_TILE_DIM][IN_TILE_DIM];
  if (row >= 0 && row < height && col >= 0 && col < width) {
    N_s[threadIdx.y][threadIdx.x] = N[row * width + col];
  } else {
    N_s[threadIdx.y][threadIdx.x] = 0.0f;
  }
  __syncthreads();

  int tileCol = (int)threadIdx.x - FILTER_RADIUS;
  int tileRow = (int)threadIdx.y - FILTER_RADIUS;
  if (col >= 0 && col < width && row >= 0 && row < height) {
    if (tileCol >= 0 && tileCol < OUT_TILE_DIM && tileRow >= 0 && tileRow < OUT_TILE_DIM) {
      float Pvalue = 0.0f;
      for (int fRow = 0; fRow < FILTER_DIM; fRow++) {
        for (int fCol = 0; fCol < FILTER_DIM; fCol++) {
          Pvalue += F_c[fRow][fCol] * N_s[tileRow + fRow][tileCol + fCol];
        }
      }
      P[row * width + col] = Pvalue;
    }
  }
}

__global__ void convolution_cached_tiled_2D_const_mem_kernel(const float* N,
                                                             float* P,
                                                             int width,
                                                             int height) {
  int col = (int)(blockIdx.x * TILE_DIM + threadIdx.x);
  int row = (int)(blockIdx.y * TILE_DIM + threadIdx.y);

  __shared__ float N_s[TILE_DIM][TILE_DIM];
  if (row < height && col < width) {
    N_s[threadIdx.y][threadIdx.x] = N[row * width + col];
  } else {
    N_s[threadIdx.y][threadIdx.x] = 0.0f;
  }
  __syncthreads();

  if (col < width && row < height) {
    float Pvalue = 0.0f;
    for (int fRow = 0; fRow < FILTER_DIM; fRow++) {
      for (int fCol = 0; fCol < FILTER_DIM; fCol++) {
        int sharedCol = (int)threadIdx.x - FILTER_RADIUS + fCol;
        int sharedRow = (int)threadIdx.y - FILTER_RADIUS + fRow;
        bool insideTile =
            sharedCol >= 0 && sharedCol < TILE_DIM && sharedRow >= 0 && sharedRow < TILE_DIM;

        if (insideTile) {
          Pvalue += F_c[fRow][fCol] * N_s[sharedRow][sharedCol];
        } else {
          int inRow = row - FILTER_RADIUS + fRow;
          int inCol = col - FILTER_RADIUS + fCol;
          if (inRow >= 0 && inRow < height && inCol >= 0 && inCol < width) {
            Pvalue += F_c[fRow][fCol] * N[inRow * width + inCol];
          }
        }
      }
    }
    P[row * width + col] = Pvalue;
  }
}

const int IMAGE_WIDTH = BY_BACKEND(4096, 500);
const int IMAGE_HEIGHT = BY_BACKEND(4096, 500);
const int TIMED_LAUNCHES_AFTER_CHECK = BY_BACKEND(20, 0);
const int BASIC_BLOCK_DIM = 16;
const unsigned RANDOM_SEED = 42;
const double MAX_RELATIVE_ERROR = 1e-4;

const int FLOP_PER_TAP = 2;
const int BYTES_PER_FLOAT = sizeof(float);
const double H100_FP32_FLOP_PER_SECOND = 66.9e12;
const double H100_DRAM_BYTES_PER_SECOND = 3.35e12;
const double H100_RIDGE_FLOP_PER_BYTE = H100_FP32_FLOP_PER_SECOND / H100_DRAM_BYTES_PER_SECOND;

const int TABLE_OUTPUT_TILES[] = {8, 16, 28, 32, 64};
const int TABLE_FILTER_DIMS[] = {3, 5, 7, 9, 11, 13};

using Image = std::vector<float>;
using FilterWeights = std::vector<float>;

struct Problem {
  int width;
  int height;
  Image image;
  FilterWeights weights;
  Filter filter;
};

struct DeviceBuffers {
  float* image;
  float* weights;
  float* output;
  size_t imageBytes;
};

struct Reference {
  Image output;
  double ms;
};

static size_t pixelCount(const Problem& problem) {
  return (size_t)problem.width * problem.height;
}

static Image randomImage(int width, int height, std::mt19937& rng) {
  std::uniform_real_distribution<float> pixelValue(0.0f, 1.0f);
  Image image((size_t)width * height);
  for (float& pixel : image) {
    pixel = pixelValue(rng);
  }
  return image;
}

static FilterWeights randomWeights(std::mt19937& rng) {
  std::uniform_real_distribution<float> weightValue(-1.0f, 1.0f);
  FilterWeights weights(FILTER_DIM * FILTER_DIM);
  for (float& weight : weights) {
    weight = weightValue(rng);
  }
  return weights;
}

static Filter asFilterStruct(const FilterWeights& weights) {
  Filter filter;
  for (int row = 0; row < FILTER_DIM; ++row) {
    for (int col = 0; col < FILTER_DIM; ++col) {
      filter.w[row][col] = weights[row * FILTER_DIM + col];
    }
  }
  return filter;
}

static Problem makeProblem() {
  std::mt19937 rng(RANDOM_SEED);

  Problem problem;
  problem.width = IMAGE_WIDTH;
  problem.height = IMAGE_HEIGHT;
  problem.image = randomImage(problem.width, problem.height, rng);
  problem.weights = randomWeights(rng);
  problem.filter = asFilterStruct(problem.weights);
  return problem;
}

static void convolution_cpu(const std::vector<float>& N,
                            const std::vector<float>& F,
                            int r,
                            std::vector<float>& P,
                            int width,
                            int height) {
  const int m = 2 * r + 1;
  for (int outRow = 0; outRow < height; ++outRow) {
    for (int outCol = 0; outCol < width; ++outCol) {
      float Pvalue = 0.0f;
      for (int fRow = 0; fRow < m; ++fRow) {
        for (int fCol = 0; fCol < m; ++fCol) {
          int inRow = outRow - r + fRow;
          int inCol = outCol - r + fCol;
          if (inRow >= 0 && inRow < height && inCol >= 0 && inCol < width) {
            Pvalue += F[fRow * m + fCol] * N[inRow * width + inCol];
          }
        }
      }
      P[outRow * width + outCol] = Pvalue;
    }
  }
}

static Reference computeReference(const Problem& problem) {
  Reference reference;
  reference.output.resize(pixelCount(problem));

  Timer timer;
  convolution_cpu(problem.image,
                  problem.weights,
                  FILTER_RADIUS,
                  reference.output,
                  problem.width,
                  problem.height);
  reference.ms = timer.ms();
  return reference;
}

static double maxRelativeError(const Image& expected, const Image& actual) {
  double worst = 0.0;
  for (size_t i = 0; i < expected.size(); ++i) {
    double difference = std::fabs((double)expected[i] - (double)actual[i]);
    double error = difference / (1.0 + std::fabs((double)expected[i]));
    if (error > worst) {
      worst = error;
    }
  }
  return worst;
}

static DeviceBuffers uploadToDevice(const Problem& problem) {
  DeviceBuffers device;
  device.imageBytes = pixelCount(problem) * sizeof(float);
  const size_t weightBytes = problem.weights.size() * sizeof(float);

  CHECK(cudaMalloc(&device.image, device.imageBytes));
  CHECK(cudaMalloc(&device.weights, weightBytes));
  CHECK(cudaMalloc(&device.output, device.imageBytes));

  CHECK(cudaMemcpy(device.image, problem.image.data(), device.imageBytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(device.weights, problem.weights.data(), weightBytes, cudaMemcpyHostToDevice));
  CHECK(cudaMemcpyToSymbol(F_c, problem.weights.data(), weightBytes));
  return device;
}

static void freeDevice(DeviceBuffers& device) {
  CHECK(cudaFree(device.image));
  CHECK(cudaFree(device.weights));
  CHECK(cudaFree(device.output));
}

class KernelTable {
 public:
  KernelTable(const DeviceBuffers& device, const Problem& problem, const Image& expected)
      : device_(device), expected_(expected), actual_(expected.size()) {
    const double filterTaps = FILTER_DIM * FILTER_DIM;
    flop_ = FLOP_PER_TAP * filterTaps * (double)problem.width * problem.height;
    bytesToReadAndWriteOnce_ = 2.0 * device.imageBytes;
  }

  void printHeader(double cpuMs) const {
    std::printf("\n%-34s %10s %10s %10s  %s\n", "kernel", "ms", "GFLOP/s", "GB/s*", "check");
    std::printf("%-34s %10.3f %10.2f %10s  %s\n",
                "CPU reference (1 thread)",
                cpuMs,
                gflopPerSecond(cpuMs),
                "-",
                "-");
  }

  template <class LaunchKernel>
  void run(const char* name, LaunchKernel launchKernel) {
    CHECK(cudaMemset(device_.output, 0, device_.imageBytes));
    Timer timer;
    launchKernel();
    CHECK(cudaGetLastError());
    double ms = timer.ms();

    CHECK(cudaMemcpy(actual_.data(), device_.output, device_.imageBytes, cudaMemcpyDeviceToHost));
    const double error = maxRelativeError(expected_, actual_);
    const bool passed = error < MAX_RELATIVE_ERROR;
    allPassed_ = allPassed_ && passed;

    if (TIMED_LAUNCHES_AFTER_CHECK > 0) {
      ms = averageMsOfRepeatedLaunches(launchKernel);
    }
    printRow(name, ms, passed, error);
  }

  bool allPassed() const {
    return allPassed_;
  }

 private:
  template <class LaunchKernel>
  static double averageMsOfRepeatedLaunches(LaunchKernel launchKernel) {
    Timer timer;
    for (int i = 0; i < TIMED_LAUNCHES_AFTER_CHECK; ++i) {
      launchKernel();
    }
    return timer.ms() / TIMED_LAUNCHES_AFTER_CHECK;
  }

  double gflopPerSecond(double ms) const {
    return flop_ / (ms * 1e6);
  }

  double gigabytesPerSecond(double ms) const {
    return bytesToReadAndWriteOnce_ / (ms * 1e6);
  }

  void printRow(const char* name, double ms, bool passed, double error) const {
    std::printf("%-34s %10.3f %10.2f %10.2f  %s (max rel err %.1e)\n",
                name,
                ms,
                gflopPerSecond(ms),
                gigabytesPerSecond(ms),
                passed ? "PASS" : "FAIL",
                error);
  }

  const DeviceBuffers& device_;
  const Image& expected_;
  Image actual_;
  double flop_;
  double bytesToReadAndWriteOnce_;
  bool allPassed_ = true;
};

static void runEveryKernel(KernelTable& table,
                           const DeviceBuffers& device,
                           const Problem& problem) {
  const float* N = device.image;
  const float* F = device.weights;
  float* P = device.output;
  const int width = problem.width;
  const int height = problem.height;

  const dim3 basicBlock(BASIC_BLOCK_DIM, BASIC_BLOCK_DIM);
  const dim3 basicGrid(cdiv(width, BASIC_BLOCK_DIM), cdiv(height, BASIC_BLOCK_DIM));
  const dim3 inputTileBlock(IN_TILE_DIM, IN_TILE_DIM);
  const dim3 outputTileGrid(cdiv(width, OUT_TILE_DIM), cdiv(height, OUT_TILE_DIM));
  const dim3 cachedTileBlock(TILE_DIM, TILE_DIM);
  const dim3 cachedTileGrid(cdiv(width, TILE_DIM), cdiv(height, TILE_DIM));

  table.run("(a)  basic, filter in global", [&] {
    launch(
        convolution_2D_basic_kernel, basicGrid, basicBlock, N, F, P, FILTER_RADIUS, width, height);
  });
  table.run("(a') basic, filter as struct arg", [&] {
    launch(
        convolution_2D_struct_kernel, basicGrid, basicBlock, N, problem.filter, P, width, height);
  });
  table.run("(b)  constant-memory filter", [&] {
    launch(
        convolution_2D_const_mem_kernel, basicGrid, basicBlock, N, P, FILTER_RADIUS, width, height);
  });
  table.run("(c)  tiled, block = input tile", [&] {
    launch(
        convolution_tiled_2D_const_mem_kernel, outputTileGrid, inputTileBlock, N, P, width, height);
  });
  table.run("(d)  tiled, cached halo", [&] {
    launch(convolution_cached_tiled_2D_const_mem_kernel,
           cachedTileGrid,
           cachedTileBlock,
           N,
           P,
           width,
           height);
  });
}

static void printTimingFootnote() {
  std::printf("* GB/s counts only the minimum traffic (read the image once, write it once).\n");
  std::printf("  In CPU emulation the times measure thread scheduling, not memory behaviour.\n");
}

static void printInputTileUsage() {
  const int blockThreads = IN_TILE_DIM * IN_TILE_DIM;
  const int outputsPerTile = OUT_TILE_DIM * OUT_TILE_DIM;
  const double computingPercent = 100.0 * outputsPerTile / blockThreads;

  std::printf("\nKernel (c): block %dx%d = %d threads, output tile %dx%d = %d (%.1f%% compute)\n",
              IN_TILE_DIM,
              IN_TILE_DIM,
              blockThreads,
              OUT_TILE_DIM,
              OUT_TILE_DIM,
              outputsPerTile,
              computingPercent);
}

static double tiledKernelIntensity(int outputTile, int filterDim) {
  const int inputTile = outputTile + filterDim - 1;
  const double flop = (double)FLOP_PER_TAP * outputTile * outputTile * filterDim * filterDim;
  const double loadedElements = (double)inputTile * inputTile;
  const double storedElements = (double)outputTile * outputTile;
  return flop / (BYTES_PER_FLOAT * (loadedElements + storedElements));
}

static double idealIntensity(int filterDim) {
  return filterDim * filterDim / 4.0;
}

static char boundOnH100(double intensity) {
  const char COMPUTE_BOUND = 'C';
  const char MEMORY_BOUND = 'M';
  return intensity >= H100_RIDGE_FLOP_PER_BYTE ? COMPUTE_BOUND : MEMORY_BOUND;
}

static void printIntensityCell(double intensity) {
  std::printf("  %6.2f%c ", intensity, boundOnH100(intensity));
}

static void printIntensityTableHeader() {
  std::printf(
      "\nArithmetic intensity of the tiled kernel (FLOP/B), output tile t x t, filter m x m\n");
  std::printf("loads of the input tile + stores of the output tile; H100 ridge = %.1f FLOP/B\n",
              H100_RIDGE_FLOP_PER_BYTE);
  std::printf("(M = memory-bound, C = compute-bound on H100)\n");

  std::printf("  t \\ m ");
  for (int filterDim : TABLE_FILTER_DIMS) {
    char label[16];
    std::snprintf(label, sizeof label, "%dx%d", filterDim, filterDim);
    std::printf("%9s ", label);
  }
  std::printf("\n");
}

static void printIntensityTable() {
  printIntensityTableHeader();

  for (int outputTile : TABLE_OUTPUT_TILES) {
    std::printf("  %5d ", outputTile);
    for (int filterDim : TABLE_FILTER_DIMS) {
      printIntensityCell(tiledKernelIntensity(outputTile, filterDim));
    }
    std::printf("\n");
  }

  std::printf("  ideal ");
  for (int filterDim : TABLE_FILTER_DIMS) {
    printIntensityCell(idealIntensity(filterDim));
  }
  std::printf("   (m^2/4: every input read once, every output written once)\n");
}

int main() {
  std::printf("2D convolution, %d x %d image, %dx%d filter, backend: %s\n",
              IMAGE_WIDTH,
              IMAGE_HEIGHT,
              FILTER_DIM,
              FILTER_DIM,
              CUDA_BACKEND);

  const Problem problem = makeProblem();
  const Reference reference = computeReference(problem);
  DeviceBuffers device = uploadToDevice(problem);

  KernelTable table(device, problem, reference.output);
  table.printHeader(reference.ms);
  runEveryKernel(table, device, problem);
  printTimingFootnote();

  printInputTileUsage();
  printIntensityTable();

  freeDevice(device);
  std::printf("\n%s\n", table.allPassed() ? "ALL PASS" : "SOME KERNEL FAILED");
  return table.allPassed() ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. conv.cu -o conv && ./conv

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/05_conv/README.md`.

**Next:** open `ec2/06_stencil/colab_1_step_by_step.ipynb`.